# Clase 3 — NLP aplicado: tokens, embeddings, clasificación y entidades clínicas

## Pregunta central

> **¿Cómo convertimos un texto en algo que una computadora pueda entender y usar?**

## Idea principal

Un texto es solo una secuencia de caracteres. Para que una aplicación trabaje con él hay que darle tres pasos: dividirlo en **tokens** (piezas), representarlo como **vectores** (embeddings) y luego **clasificarlo** (¿de qué trata?) y **extraerle datos** (¿qué contiene?). En esta clase entendemos cada paso con calma y al final hacemos un laboratorio pequeño de extracción.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Explicar qué es un token y por qué los modelos no leen palabras enteras.
- Explicar qué es un embedding y qué significa "espacio vectorial".
- Entender que hay varias herramientas para clasificar y extraer, y que elegimos la más fácil de trabajar.
- Clasificar un mensaje simple y extraerle datos con reglas y diccionarios.
- Reconocer cuándo hace falta que una persona revise el resultado.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | ¿Qué es un token? |
| 2 | ¿Qué es un embedding? El espacio vectorial |
| 3 | Clasificar: elegir la herramienta más fácil |
| 4 | Extraer entidades: reglas y diccionarios |
| 5 | Laboratorio: extraer datos de un mensaje |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario.

**Conexión con el programa:** la clase 2 terminó con texto transcrito por Whisper. Esta clase explica cómo ese texto se convierte en datos que una aplicación puede usar.


## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Token | Pieza en la que se divide un texto (palabra, fragmento o signo) |
| Tokenizador | Herramienta que divide el texto en tokens |
| Vocabulario | Conjunto de tokens que un modelo conoce |
| Embedding | Vector (lista de números) que representa el contenido de un texto |
| Espacio vectorial | "Mapa" donde cada texto es un punto; los parecidos quedan cerca |
| Similitud coseno | Regla para medir qué tan cerca están dos puntos |
| Clasificar | Decidir de qué categoría es un texto |
| Ancla | Frase de ejemplo que describe una categoría |
| Entidad | Dato concreto dentro del texto (fecha, fármaco, dosis) |
| Gazetteer | Diccionario de términos conocidos de un dominio |
| Regex | Patrón de búsqueda en texto |
| NER | Extracción de entidades con modelos entrenados |
| Dato estructurado | Información con campos definidos que una aplicación puede consumir |
| JSON | Formato de texto para datos estructurados |
| Validación humana | Revisión de una persona antes de usar el resultado |


In [ ]:
# --- Preparación: imports y configuración ---
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Semilla fija: los números aleatorios salen iguales en cada ejecución.
SEED = 42

print("Entorno listo.")


---
## 1. ¿Qué es un token?

Un modelo no lee palabras: lee **tokens**, piezas de texto que aparecen en su **vocabulario**.

> **Analogía del LEGO.** Las palabras son piezas armadas; los tokens son los bloques. La palabra "ibuprofeno" puede venir en un solo bloque o en varios, según el vocabulario del modelo. El modelo nunca ve la palabra armada: ve los bloques y su orden.

¿Por qué no trabajar directamente con palabras?

| Problema de usar palabras | Qué pasa |
|---|---|
| Vocabulario abierto | Aparecen palabras nuevas todo el tiempo (nombres, marcas, fármacos) |
| Palabras raras | "hipotiroidismo" puede aparecer 2 veces en millones de textos |
| Formas flexivas | "tomo", "tomó", "tomando" serían 3 palabras distintas |
| Errores y tipeos | "ibuprfeno" sería una palabra desconocida |

La solución son los **tokens de subpalabra**: fragmentos frecuentes que se recombinan. Con pocas piezas se puede escribir cualquier texto, incluso con errores o palabras inventadas. Ninguna palabra queda "fuera del sistema".

> **Pensalo así:** el tokenizador es la primera capa de cualquier modelo de lenguaje. Todo lo que el modelo "lee" pasa por ahí.


In [ ]:
# --- Ver un texto dividido en tokens ---
# Usamos un tokenizador real (el de los modelos GPT) solo para ver
# cómo se divide un texto. No hace falta entender cómo funciona por dentro.
import tiktoken

tokenizador = tiktoken.get_encoding("cl100k_base")

FRASE = "El paciente toma ibuprofeno 600 mg cada 8 horas por dolor lumbar."

# Dividimos la frase en tokens y mostramos cada pieza.
ids = tokenizador.encode(FRASE)
tokens = [tokenizador.decode([i]) for i in ids]

print("Frase:", FRASE)
print(f"\nSe divide en {len(tokens)} tokens:")
print(tokens)


### Qué observar

- La frase se parte en piezas: algunas son palabras completas y otras son fragmentos.
- Mirá cómo quedó "ibuprofeno": una palabra larga puede dividirse en varios tokens.
- El número de tokens importa porque define cuánto "espacio" ocupa un texto (lo veremos en la clase 4).

> **Pregunta de interpretación:** ¿por qué creés que conviene dividir "ibuprofeno" en fragmentos en vez de guardarla como una sola pieza?


---
## 2. ¿Qué es un embedding? El espacio vectorial

Un **embedding** es una lista de números (un **vector**) que representa el contenido de un texto. Cada texto se convierte en un punto dentro de un "mapa" llamado **espacio vectorial**.

> **Analogía del mapa de ciudades.** Imaginá un mapa donde cada frase es una ciudad. Las frases que significan cosas parecidas quedan en ciudades cercanas; las que significan cosas distintas, lejos. "pedir turno" y "sacar cita" quedan vecinas; "pedir turno" y "dolor de cabeza", lejos.

```text
texto -> tokenizador -> tokens -> encoder -> vector (embedding)
```

El **encoder** es la herramienta que convierte el texto en el vector. Para comparar dos textos, medimos qué tan cerca están sus puntos en el mapa. Eso se llama **similitud coseno**: un número que dice cuánto apuntan en la misma dirección.

> **Pensalo así:** no necesitás entender cómo el encoder arma el vector. Solo necesitás saber que textos parecidos quedan cerca, y que podemos medir esa cercanía con un número.


In [ ]:
# --- Ver el espacio vectorial con un ejemplo simplificado ---
# Para poder "ver" el mapa, usamos vectores de 2 números (2 dimensiones).
# En la realidad los embeddings tienen cientos de números, pero la idea
# es la misma: textos parecidos quedan cerca.

# Cada frase es un punto (x, y) en el mapa.
frases = {
    "pedir turno": (0.9, 0.8),
    "sacar cita": (0.8, 0.9),
    "dolor de cabeza": (0.1, 0.2),
    "pregunta de receta": (0.2, 0.1),
}

fig, ax = plt.subplots(figsize=(6, 6))
for nombre, (x, y) in frases.items():
    ax.scatter(x, y, s=200)
    ax.annotate(nombre, (x, y), textcoords="offset points", xytext=(6, 6), fontsize=10)

ax.set_xlim(-0.2, 1.2)
ax.set_ylim(-0.2, 1.2)
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_aspect("equal")
ax.set_title("Espacio vectorial (simplificado a 2 dimensiones)")
ax.set_xlabel("dimensión 1")
ax.set_ylabel("dimensión 2")
plt.grid(alpha=0.3)
plt.show()


### Qué observar

- "pedir turno" y "sacar cita" quedan cerca: significan lo mismo aunque usan palabras distintas.
- "dolor de cabeza" y "pregunta de receta" quedan lejos de las de turnos.
- Este mapa es **simplificado** (2 dimensiones para poder dibujarlo). Un embedding real tiene cientos de dimensiones, pero la idea es la misma.

> **Pregunta de interpretación:** ¿qué dos frases del mapa están más cerca? ¿Tiene sentido que estén juntas?

> **Importante:** la cercanía depende del encoder y de los datos con los que fue entrenado. No es "el significado verdadero", es la cercanía que aprendió ese modelo.


---
## 3. Clasificar: elegir la herramienta más fácil

Ahora que el texto se puede representar, queremos **clasificarlo**: decidir de qué categoría es. Por ejemplo, un mensaje de un consultorio puede ser de **turno**, **receta**, **consulta médica** u **otro**.

### ¿Qué significa clasificar?

Clasificar es **asignar una etiqueta** a un texto. Es lo mismo que hace una persona cuando lee un mensaje y decide "esto es un pedido de turno" o "esto es una pregunta sobre un medicamento". La computadora hace lo mismo, pero necesita una regla para decidir.

> **Analogía del recepcionista.** Un recepcionista nuevo recibe mensajes y debe ordenarlos en carpetas: "turnos", "recetas", "consultas médicas", "otros". Al principio no sabe qué hacer, así que le damos instrucciones. Esas instrucciones son la "herramienta" de clasificación.

### Las herramientas disponibles

Existen varias formas de darle esas instrucciones a la computadora:

| Herramienta | Cómo funciona | Dificultad | Cuándo conviene |
|---|---|---|---|
| Reglas manuales | Si el texto contiene "turno", es de turnos | Muy fácil, pero frágil | Casos muy simples y fijos |
| Matching por similitud | Compara el mensaje con frases de ejemplo | Fácil, no necesita datos | Empezar rápido, sin datos |
| Clasificador entrenado | Aprende de ejemplos etiquetados | Más potente, necesita datos | Cuando hay muchos ejemplos |
| Modelo de lenguaje (LLM) | Preguntamos al modelo | Potente, pero más pesado | Cuando ya usamos un LLM |

### ¿Cuál elegimos y por qué?

> **Pensalo así:** no siempre hace falta la herramienta más potente. Para un problema simple, la herramienta más fácil de trabajar suele ser la mejor. Elegir la herramienta correcta es parte del trabajo de un programador: no se trata de usar lo más avanzado, sino lo que resuelve el problema con el menor costo.

En esta clase nos quedamos con el **matching por similitud**. ¿Por qué?

- **No necesita datos etiquetados:** solo escribimos frases de ejemplo por categoría.
- **Es fácil de entender:** el mensaje se compara con cada frase y se elige la más parecida.
- **Es fácil de mantener:** si una categoría cambia, cambiamos su frase de ejemplo.

### Cómo funciona el matching por similitud

```text
mensaje -> se compara con cada ancla -> se elige la más parecida -> categoría
```

Cada categoría tiene una **ancla**: una frase de ejemplo que la describe. El mensaje se convierte en un embedding y se compara con el embedding de cada ancla. El ancla más parecida gana.

> **Analogía del mapa otra vez.** Cada ancla es una ciudad en el mapa. El mensaje es un punto nuevo. Buscamos la ciudad más cercana a ese punto: esa es la categoría.

> **Importante:** la calidad depende de qué tan bien escribamos las anclas. Si el ancla de "receta" está mal escrita, los mensajes de receta se van a otra categoría. Las anclas son la "instrucción" que le damos al sistema.


In [ ]:
# --- Clasificar un mensaje por similitud (herramienta fácil) ---
# Definimos una frase "ancla" por categoría. El mensaje se compara
# con cada ancla y se elige la más parecida.
ANCLAS = {
    "turno": "quiero pedir, cambiar o cancelar un turno o cita médica",
    "receta": "pregunta sobre un medicamento, dosis o receta",
    "consulta_medica": "descripción de un síntoma o dolor actual",
    "otro": "consulta administrativa sobre horarios, dirección o pagos",
}

# Para comparar usamos un encoder de texto real (all-MiniLM-L6-v2).
# Si no está disponible, usamos una comparación por palabras compartidas.
try:
    from sentence_transformers import SentenceTransformer

    encoder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device="cpu"
    )
    modo = "embeddings"
except Exception:
    encoder = None
    modo = "palabras compartidas (fallback)"

def similitud(a, b):
    """Devuelve un número que dice qué tan parecidos son dos textos."""
    if encoder is not None:
        va = encoder.encode([a], normalize_embeddings=True)[0]
        vb = encoder.encode([b], normalize_embeddings=True)[0]
        return float(va @ vb)
    # Fallback simple: fracción de palabras en común.
    pa = set(a.lower().split())
    pb = set(b.lower().split())
    return len(pa & pb) / max(1, len(pa | pb))

def clasificar(mensaje):
    """Elige la categoría más parecida al mensaje."""
    resultados = [
        (categoria, similitud(mensaje, ancla))
        for categoria, ancla in ANCLAS.items()
    ]
    resultados.sort(key=lambda par: -par[1])
    return resultados[0]

mensaje = "Hola, me quedó muy lejos el turno del martes, ¿lo puedo pasar al jueves?"
categoria, score = clasificar(mensaje)
print(f"Mensaje: {mensaje}")
print(f"\nModo: {modo}")
print(f"Categoría elegida: {categoria}  (similitud {score:.2f})")


### Qué observar

- El mensaje se comparó con cada ancla y se eligió la más parecida.
- No hizo falta entrenar nada: solo escribimos frases de ejemplo por categoría.
- La calidad depende de qué tan bien describamos cada categoría con su ancla.

> **Pregunta de interpretación:** ¿qué pasaría si la ancla de "receta" estuviera mal escrita? ¿A dónde irían los mensajes de receta?

> **Pensalo así:** esta herramienta es fácil de trabajar porque no necesita datos etiquetados. Cuando el problema crece, se puede pasar a un clasificador entrenado, pero para empezar alcanza.


---
## 4. Extraer entidades: reglas y diccionarios

Clasificar dice **de qué trata** el mensaje. **Extraer entidades** responde **qué datos concretos** contiene: fechas, horas, medicamentos, especialidades, dosis.

### ¿Qué significa extraer entidades?

Una **entidad** es un dato concreto dentro del texto. Por ejemplo, en "Tomo metformina 850 mg y quiero cambiar el turno del martes", las entidades son:

- **medicamento:** metformina
- **dosis:** 850 mg
- **día:** martes

> **Analogía del formulario.** Clasificar elige la carpeta donde va el mensaje; extraer entidades llena los campos del formulario: "medicamento: metformina", "dosis: 850 mg", "día: martes". La aplicación necesita esos campos para trabajar.

### Las herramientas disponibles

También hay varias formas de extraer entidades:

| Herramienta | Cómo funciona | Dificultad | Cuándo conviene |
|---|---|---|---|
| Diccionarios (gazetteer) | Listas de términos conocidos | Muy fácil | Vocabulario cerrado: fármacos, especialidades |
| Reglas (regex) | Patrones de texto: números, fechas, horas | Fácil | Datos con formato predecible |
| Modelos de NER | Red entrenada para etiquetar tokens | Más potente, más pesada | Vocabulario abierto y texto ruidoso |

> NER significa Named Entity Recognition (Reconocimiento de Entidades Nombradas). Es una herramienta que extrae entidades de un texto usando un modelo entrenado, en lugar de usar diccionarios o reglas.

### ¿Cuál elegimos y por qué?

> **Pensalo así:** igual que con la clasificación, elegimos la herramienta más fácil de trabajar. En salud, los diccionarios clínicos (fármacos, especialidades, síntomas) son muy útiles porque el vocabulario es acotado: sabemos qué términos buscar.

En esta clase combinamos **diccionarios** y **reglas**:

- **Diccionario (gazetteer):** una lista de términos que conocemos. Buscamos si el texto contiene alguno. Es como buscar en una lista de compras.
- **Regla (regex):** un patrón de texto. Por ejemplo, "un número seguido de mg" es una dosis. Es como buscar algo que "tiene forma de" dosis.

### Cómo funciona

```text
texto -> buscar términos del diccionario -> entidades (medicamentos, días)
texto -> buscar patrones con regex -> entidades (dosis, horas)
```

> **Analogía del buscador de tesoros.** El diccionario es como buscar tesoros que ya conocemos por nombre ("metformina", "martes"). La regla es como buscar tesoros que reconocemos por su forma ("un número + mg"). Juntos cubren la mayoría de los datos útiles.

> **Importante:** estas herramientas encuentran lo **predecible**. Lo que se escapa: "la semana que viene", "desde ayer", sinónimos, errores de transcripción. Cuando el vocabulario es abierto o el texto es ruidoso, se necesitan modelos de NER, pero para empezar alcanza.


In [ ]:
# --- Extraer entidades con diccionarios y reglas ---
# Diccionario (gazetteer): términos conocidos del dominio.
MEDICAMENTOS = ["ibuprofeno", "paracetamol", "losartán", "metformina", "insulina"]
ESPECIALIDADES = ["dermatóloga", "cardióloga", "dentista", "clínica"]
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

def extraer_entidades(texto):
    """Busca datos conocidos dentro del texto."""
    texto_normalizado = texto.lower()
    entidades = {
        "medicamentos": [m for m in MEDICAMENTOS if m in texto_normalizado],
        "especialidades": [e for e in ESPECIALIDADES if e in texto_normalizado],
        "dias": [d for d in DIAS if d in texto_normalizado],
        "dosis": re.findall(r"\d+\s*(?:mg|ml)", texto_normalizado),
        "horas": re.findall(r"\d+\s*(?:hs|h)", texto_normalizado),
    }
    # Dejamos solo los campos que encontraron algo.
    return {k: v for k, v in entidades.items() if v}

mensaje = "Tomo metformina 850 mg y quiero cambiar el turno del martes."
print("Mensaje:", mensaje)
print("Entidades encontradas:", extraer_entidades(mensaje))


### Qué observar

- Las reglas y diccionarios encuentran lo **predecible**: dosis, días, términos conocidos.
- Lo que se escapa: "la semana que viene", "desde ayer", sinónimos, errores de transcripción.

> **Pregunta de interpretación:** ¿qué pasa si Whisper transcribe "metformina" como "metformna"? ¿Lo encuentra el diccionario?

> **Pensalo así:** las reglas y diccionarios son baratas, explicables y fáciles de mantener. Cuando el vocabulario es abierto o el texto es ruidoso, se necesitan modelos de NER, pero para empezar alcanza.


---
## 5. Laboratorio — clasificar mensajes y mejorar las anclas

Ahora juntamos todo en un laboratorio pequeño. Vamos a clasificar varios mensajes y a **mejorar las anclas** cuando el sistema se equivoque.

> **Duración sugerida:** 20 minutos.

### El caso

La recepción recibe mensajes de pacientes. Queremos que el sistema clasifique cada mensaje en una categoría (**turno**, **receta**, **consulta médica** u **otro**), para que una persona los revise.

### Los mensajes a clasificar

Te damos varios mensajes, algunos **contradictorios** (mezclan temas o son ambiguos). El sistema debe decidir la categoría de cada uno:

```text
1. "Hola, me recetaron losartán 50 mg y quiero saber si lo tomo en ayunas."
2. "Tengo dolor de cabeza fuerte desde ayer, ¿qué puedo tomar?"
3. "¿Me dan un turno con la dermatóloga para el viernes?"
4. "El ibuprofeno me cae mal al estómago, ¿puedo cambiar mi turno?"
5. "¿A qué hora abren los sábados?"
6. "Necesito que me renueven la receta de la insulina."
```

### Qué hacer

1. Ejecutá la celda de abajo y observá qué categoría elige el sistema para cada mensaje.
2. Buscá los mensajes donde el sistema **se equivocó** o dudó.
3. **Modificá las anclas** (las frases de ejemplo de cada categoría) para mejorar el resultado y volvé a ejecutar.
4. Agregá un término a los diccionarios de entidades y probá de nuevo.

> **Pensalo así:** las anclas son la "instrucción" que le damos al sistema. Si un mensaje se clasifica mal, la solución suele estar en mejorar el ancla de esa categoría, no en cambiar el código.

In [ ]:
# ✏️ LABORATORIO: clasificar varios mensajes y mejorar las anclas.
# TODO 1: modificá las anclas de abajo para mejorar la clasificación.
ANCLAS = {
    "turno": "cita médica",
    "receta": "receta",
    "consulta_medica": "dolor",
    "otro": "consulta administrativa",
}

# TODO 2: agregá o cambiá mensajes para probar casos contradictorios.
MENSAJES_LAB = [
    "Hola, me recetaron losartán 50 mg y quiero saber si lo tomo en ayunas.",
    "Tengo dolor de cabeza fuerte desde ayer, ¿qué puedo tomar?",
    "¿Me dan un turno con la dermatóloga para el viernes?",
    "El ibuprofeno me cae mal al estómago, ¿puedo cambiar mi turno?",
    "¿A qué hora abren los sábados?",
    "Necesito que me renueven la receta de la insulina.",
]

# Clasificamos cada mensaje y mostramos el resultado.
print("Clasificación de los mensajes:")
for mensaje in MENSAJES_LAB:
    categoria, score = clasificar(mensaje)
    print(f"  [{categoria:15s} {score:.2f}] {mensaje}")

# Extraemos las entidades del primer mensaje como ejemplo.
print("\nEntidades del primer mensaje:")
print(extraer_entidades(MENSAJES_LAB[0]))

### Para compartir al final

1. Mostrá qué mensajes clasificó bien el sistema y cuáles no.
2. Indicá qué ancla modificaste y cómo cambió el resultado.
3. Mostrá un mensaje contradictorio y explicá por qué fue difícil de clasificar.
4. Explicá en qué caso marcarías un mensaje para revisión humana.

> **Cierre:** el sistema no reemplaza a la recepcionista: le entrega el trabajo ordenado. Las anclas son la "instrucción" que le damos al sistema, y mejorarlas es parte del trabajo de un programador.

---

## Síntesis de la clase

- Un **token** es una pieza en la que se divide un texto; los modelos no leen palabras enteras.
- Un **embedding** es un vector que ubica un texto en un espacio vectorial; los parecidos quedan cerca.
- Para clasificar y extraer hay varias herramientas; elegimos la más fácil de trabajar.
- Clasificamos por similitud con frases de ejemplo; extraemos con diccionarios y reglas.
- El resultado es un registro estructurado que una aplicación puede usar.
- La decisión final siempre la toma una persona.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 4 abre la caja del LLM: cómo funciona la atención, qué es la ventana de contexto y por qué los tokens definen el costo y el límite de todo lo que vimos hoy.

## Conexión con el track

Salud usará estos conceptos para convertir transcripciones en datos estructurados: intenciones de turnos, medicación y síntomas, siempre con validación humana en decisiones clínicas.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.
